#### 1. Obtain protein IDs of orthologs per GENE from NCBI
#### 2. Using the IDS Fetch protein sequence ID from 'records' from the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) 
#### 3. Store nested dictionary of protein : sequences
#### 4. identify and extract unique sequences, human first then non-human
#### 5. Save sequences in FASTA format for each gene in /Fasta folder


#### 

In [18]:
run Kondrashov

* function: fetch protein record of each ortholog from NCBI

In [19]:
def fetch_gene_record(gene_id):
    """
    Fetch one Entrez Gene XML record and parse it with xmltodict.
    """
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="xml", retmode="text")
    record = x2d.parse(handle.read().decode("utf-8"))
    handle.close()
    return record

* function: extract protein ID from dictionary into a 'list'

In [20]:
def extract_protein_accessions(obj):
    """
    Recursively search the NCBI Gene XML dictionary and collect protein accessions.
    Returns accessions like XP_077799762.1 or NP_000148.2.
    """
    proteins = set()

    if isinstance(obj, dict):
        acc = obj.get("Gene-commentary_accession")
        ver = obj.get("Gene-commentary_version")

        if acc is not None:
            # Protein accessions usually start with NP_, XP_, or YP_
            if acc.startswith(("NP_", "XP_", "YP_")):
                if ver is not None:
                    proteins.add(f"{acc}.{ver}")
                else:
                    proteins.add(acc)

        for value in obj.values():
            proteins.update(extract_protein_accessions(value))

    elif isinstance(obj, list):
        for item in obj:
            proteins.update(extract_protein_accessions(item))

    return proteins


### [Rodentia](https://meshb.nlm.nih.gov/record/ui?ui=D012377&dcmsLinks=true)  txid9989
* RUN CODE USING DEFINED FUNCTIONS: obtain orthologs per gene 
* final output is 'records' contains gene, rodentia orthologs, details

In [21]:
xx = ['ABCD1',
'ALPL',
'AR',
'BTK',
'CASR',
'CFTR',
'CYBB',
'F7',
'F8',
'F9',
'G6PD',
'GJB1',
'HBB',
'HPRT1',
'IL2RG',
'KCNH2',
'L1CAM',
'MPZ',
'MYH7',
'PMM2',
'RHO',
'TP53',
'TTR']

In [17]:
# obtain rodentia Orthologs per gene
result = {}

for locus in xx:
    query = f"{locus}[Gene Name] AND txid9989[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    search_record = Entrez.read(handle)
    handle.close()

    gene_ids = search_record["IdList"]
    print(f"{locus}: {search_record['Count']} -> {gene_ids[:4]}")


    result[locus] = {}

    for gene_id in gene_ids:
        try:
            gene_record = fetch_gene_record(gene_id)
            protein_ids = sorted(extract_protein_accessions(gene_record))
            result[locus][gene_id] = protein_ids
            print(f"  {gene_id}: {len(protein_ids)} proteins")

            # NCBI's request limit (10/sec with an API key)
            time.sleep(0.1)

        except Exception as expt:
            print(f"  Error with {locus} / {gene_id}: {expt}")
            result[locus][gene_id] = []

ABCD1: 51 -> ['20299', '11666', '363516', '109694160']
  20299: 1 proteins
  11666: 1 proteins
  363516: 2 proteins
  109694160: 1 proteins
  103731197: 3 proteins
  101835556: 2 proteins
  100769988: 3 proteins
  311601465: 1 proteins
  308639976: 2 proteins
  308414766: 2 proteins
  308279466: 1 proteins
  306651635: 1 proteins
  143638444: 1 proteins
  142841658: 2 proteins
  138839987: 1 proteins
  131898765: 1 proteins
  130867426: 1 proteins
  129665252: 3 proteins
  128112608: 2 proteins
  127675768: 1 proteins
  127217339: 3 proteins
  127184831: 1 proteins
  126490225: 1 proteins
  125415476: 1 proteins
  125343652: 1 proteins
  124971781: 1 proteins
  124098333: 1 proteins
  122097069: 1 proteins
  119804665: 2 proteins
  118573830: 2 proteins
  117694398: 2 proteins
  116888324: 1 proteins
  116085973: 2 proteins
  114706187: 1 proteins
  114637145: 2 proteins
  114080145: 1 proteins
  113199737: 1 proteins
  110560442: 1 proteins
  110313461: 1 proteins
  110286476: 1 prote

* function: extract all protein ids per orthologs per genes into a list 

In [22]:
# create fasta folder
!mkdir fasta_lociii_rodentia

def flatten_protein_ids(variant_dict):
    """
    Convert:
        {"variant1": [ids], "variant2": [ids]}
    into one unique list of protein IDs for that gene.
    """
    seq_ids = []
    seen = set()

    for variant_id, protein_ids in variant_dict.items():
        for pid in protein_ids:
            if pid not in seen:
                seq_ids.append(pid)
                seen.add(pid)

    return seq_ids



mkdir: fasta_lociii_rodentia: File exists


* function: fetch protein details ('records') from NCBI
* modified

In [23]:
def fetch_protein_records(seq_ids):
    """
    Fetch GenBank protein records from NCBI.
    Uses chunks (at most 200) so the request does not become too large.
    """
    records = []
    chunk_size= 200
    for start in range(0, len(seq_ids), chunk_size):
        chunk = seq_ids[start:start + chunk_size]

        handle= Entrez.efetch(db="protein", rettype="gb", retmode="text", id=",".join(chunk))
        records.extend(list(SeqIO.parse(handle, "gb")))
        handle.close()

        time.sleep(0.35)

    return records

* function: get specie name from record eg 'Homo sapiens'

In [24]:
def get_species(record):
    """
    Retrieve species name from NCBI sequence record.

    Parameters
    ----------
    record : Bio.SeqRecord
        Sequence record.

    Returns
    -------
    str
        Species name.
    """
    description = record.description
    species = description.split(" [")[1][:-1]
    return species

* function; collect unique sequences; Homo sapiens first, then non Homo sapiens

In [25]:
def collect_unique_sequences_human_first(records):
    """
    Keep unique protein sequences.
    First collect unique Homo sapiens sequences,
    then collect unique non-human sequences.
    """
    seq_to_index = {}
    unique_records = []
    excluded_records = []

    inc = 0
    exc = 0

    # 1. Collect unique human sequences first
    for seq_record in records:
        sp = get_species(seq_record)
        if sp == "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}")
                inc += 1

    # 2. Collect other unique non-human sequences
    for seq_record in records:
        sp = get_species(seq_record)

        if sp != "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    return unique_records, excluded_records, inc, exc


* function; make a dict ('sequence_dict') comprising protein and its sequence

In [26]:
def make_sequence_dict(variant_dict, records):
    """
    Preserve your original nested structure, but replace each protein ID
    with its actual protein sequence.
    """
    #seq_by_id = {seq_record.id: str(seq_record.seq) for seq_record in records}
    seq_by_id = {}
    for seq_record in records:
        seq_by_id[seq_record.id: str(seq_record.seq)]= {}

    sequence_dict = {}

    for variant_id, protein_ids in variant_dict.items():
        sequence_dict[variant_id] = {}

        for pid in protein_ids:
            sequence_dict[variant_id][pid] = seq_by_id.get(pid, None)

    return sequence_dict

* RUN CODE USING CREATED FUNCTIONS
* Store dictionary containing protein id, unique sequences, etc 'summary_df' as dataframe
* save sequences in fasta file per gene

In [27]:
protein_sequences = {}
unique_records_by_gene = {}
excluded_records_by_gene = {}

summary = []

for gene, variant_dict in result.items():

    print("\n" + "=" * 80) #demarcation line
    #print(datetime.now())
    print(f"{gene} orthologs")

    # Get all protein IDs for this gene from your result dictionary
    seq_ids = flatten_protein_ids(variant_dict)

    print(f"{gene}: {len(seq_ids)} protein IDs")
    print(f"Sequence IDs: {seq_ids[:10]}{' ...' if len(seq_ids) > 10 else ''}")

    if len(seq_ids) == 0:
        protein_sequences[gene] = {}
        unique_records_by_gene[gene] = []
        excluded_records_by_gene[gene] = []

        summary.append({
            "gene": gene,
            "n_protein_ids": 0,
            "n_records_fetched": 0,
            "n_unique_sequences": 0,
            "n_excluded_duplicates": 0,
            "fasta_file": None
        })

        continue

    # Fetch protein records from NCBI
    records = fetch_protein_records(seq_ids)

    print(f"{gene}: {len(records)} protein records fetched")

    # Store nested dictionary of actual sequences
    protein_sequences[gene] = make_sequence_dict(variant_dict, records)

    # Collect unique sequences, human first
    print(f"\n{gene} orthologs: unique rodentia sequences\n")

    unique_records, excluded_records, inc, exc = collect_unique_sequences_human_first(records)

    unique_records_by_gene[gene] = unique_records
    excluded_records_by_gene[gene] = excluded_records

    # Save FASTA file for that gene
    fasta_path = f"fasta_lociii_rodentia/{gene}.fasta"

    with open(fasta_path, "w") as output:
        SeqIO.write(unique_records, output, "fasta")

    print(f"\nTotal: {inc} unique sequences, {exc} excluded")
    print(f"{fasta_path} saved!")

    summary.append({
        "gene": gene,
        "n_protein_ids": len(seq_ids),
        "n_records_fetched": len(records),
        "n_unique_sequences": inc,
        "n_excluded_duplicates": exc,
        "fasta_file": fasta_path
    })

summary_df = pd.DataFrame(summary)
summary_df


ABCD1 orthologs
ABCD1: 72 protein IDs
Sequence IDs: ['NP_033163.1', 'NP_031461.1', 'NP_001102291.1', 'XP_038955840.1', 'XP_020031513.1', 'XP_008827761.1', 'XP_008827762.1', 'XP_029417692.1', 'XP_005087005.1', 'XP_012981346.1'] ...
ABCD1: 72 protein records fetched

ABCD1 orthologs: unique rodentia sequences

0:	NP_033163.1	(92 aa)	C-C motif chemokine 22 precursor [Mus musculus]
1:	NP_031461.1	(736 aa)	ATP-binding cassette sub-family D member 1 [Mus musculus]
2:	NP_001102291.1	(737 aa)	ATP-binding cassette sub-family D member 1 [Rattus norvegicus]
3:	XP_038955840.1	(538 aa)	ATP-binding cassette sub-family D member 1 isoform X1 [Rattus norvegicus]
4:	XP_020031513.1	(739 aa)	ATP-binding cassette sub-family D member 1 [Castor canadensis]
5:	XP_008827761.1	(748 aa)	ATP-binding cassette sub-family D member 1 isoform X1 [Nannospalax galili]
6:	XP_008827762.1	(740 aa)	ATP-binding cassette sub-family D member 1 isoform X2 [Nannospalax galili]
7:	XP_029417692.1	(615 aa)	ATP-binding cassette sub

,gene,n_protein_ids,n_records_fetched,n_unique_sequences,n_excluded_duplicates,fasta_file
0,ABCD1,72,72,70,2,fasta_lociii_rodentia/ABCD1.fasta
1,ALPL,178,178,65,113,fasta_lociii_rodentia/ALPL.fasta
2,AR,76,76,71,5,fasta_lociii_rodentia/AR.fasta
3,BTK,104,104,51,53,fasta_lociii_rodentia/BTK.fasta
4,CASR,107,107,70,37,fasta_lociii_rodentia/CASR.fasta
5,CFTR,75,75,72,3,fasta_lociii_rodentia/CFTR.fasta
6,CYBB,55,55,48,7,fasta_lociii_rodentia/CYBB.fasta
7,F7,65,65,62,3,fasta_lociii_rodentia/F7.fasta
8,F8,224,224,198,26,fasta_lociii_rodentia/F8.fasta
9,F9,67,67,66,1,fasta_lociii_rodentia/F9.fasta
